# Which barriers separate high-adoption countries from low-adoption ones

The portal's barrier chart answers *what do firms complain about*. This notebook answers a
different question: **which of those complaints actually distinguishes countries where AI adoption
is high from countries where it is low.** The two have different answers, which is why this exists.

It runs step by step so each decision is visible, and ends by writing the same two CSVs that
`analyse_barriers.py` produces in one shot.

**This is association, not cause.** The barrier percentages are measured only on firms that
considered AI and declined - a group defined partly by the outcome - so nothing here says that
removing a barrier would raise adoption.

## 1 - Setup

Run from the repository root, or from `notebooks/`; the cell below finds the root either way.
Everything comes from the canonical Parquet table, not the site bundle, which is trimmed for the
browser.

In [ ]:
import sys, pathlib

ROOT = pathlib.Path.cwd()
if not (ROOT / "sme_pipeline").exists():
    ROOT = ROOT.parent                      # started inside notebooks/
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from sme_pipeline import barrier_analysis as ba

firm = pd.read_parquet(ROOT / "data" / "processed" / "firm_level.parquet")
print(f"{len(firm):,} rows in the firm-level table")
ba.SHORT_LABELS

## 2 - Build the panel

One row per country and year, for a single size tier.

Three restrictions, each load-bearing:

- **Only the three disjoint tiers.** `SME_10_249` and `ALL_GE10` contain the others, so including
  them would put the same countries in the model twice.
- **Countries only.** `EU27_2020` is an aggregate of the other rows, not another observation.
- **Barriers on `PC_ENT_AI_EC`.** As a share of *all* enterprises a barrier reads as 4-5% and the
  size bands invert, because most firms never considered AI and so answered nothing.

In [ ]:
panels = {tier: ba._panel(firm, tier) for tier in ba.TIERS}

for tier, p in panels.items():
    print(f"{ba.TIER_LABELS[tier]:18s} {len(p):3d} country-year cells | "
          f"{p.geo.nunique()} countries | years {sorted(p.time.unique())}")

panels["SMALL_10_49"].head()

### Why there is no model per country

A country has four observations - 2021, 2023, 2024, 2025 - against seven predictors. That is not a
thin model, it is not estimable. The variation that identifies a barrier's effect *is* the
variation across countries, so the model is fitted per tier and read country by country in step 7.

In [ ]:
panels["SMALL_10_49"].groupby("geo").size().describe()[["min", "50%", "max"]]

## 3 - Diagnostics before fitting

Barriers move together. The question is whether they move together *enough* to destabilise the
coefficients: a condition number above ~30, or a VIF above ~10, would be a problem.

In [ ]:
def diagnostics(panel):
    corr = panel[ba.BARRIERS].corr().to_numpy()
    ev = np.linalg.eigvalsh(corr)[::-1]
    vifs = []
    for code in ba.BARRIERS:
        others = [c for c in ba.BARRIERS if c != code]
        X = np.column_stack([np.ones(len(panel)), panel[others].to_numpy(float)])
        y = panel[code].to_numpy(float)
        beta, *_ = np.linalg.lstsq(X, y, rcond=None)
        resid = y - X @ beta
        r2 = 1 - resid @ resid / ((y - y.mean()) ** 2).sum()
        vifs.append({"barrier": ba.SHORT_LABELS[code], "VIF": round(1 / max(1 - r2, 1e-9), 1)})
    return ev[0] / ev[-1], pd.DataFrame(vifs)

for tier, p in panels.items():
    cond, vif = diagnostics(p)
    print(f"{ba.TIER_LABELS[tier]:18s} condition number {cond:5.1f}   max VIF {vif.VIF.max():.1f}")

diagnostics(panels["SMALL_10_49"])[1]

Mild - so no decorrelation step is warranted.

That retires the original idea of extracting latent factors first. PCA would also have destroyed
the thing being ranked: a component has no name, and mapping coefficients back through the
loadings reintroduces exactly the correlation it removed.

## 4 - How much do the barriers explain?

Year effects are absorbed in every model, including the baseline, so what is reported is what the
barriers add *beyond* the survey wave.

In [ ]:
for tier, p in panels.items():
    resid_y, resid_x, total_ss, base_rss = ba._prepare(p)
    base = 1 - base_rss / total_ss
    full = 1 - ba._rss(resid_y, resid_x, tuple(range(len(ba.BARRIERS)))) / total_ss
    print(f"{ba.TIER_LABELS[tier]:18s} R2 years only {base:.3f} -> with barriers {full:.3f}"
          f"   (adds {full - base:+.3f})")

## 5 - Split that between the barriers

A coefficient is not a contribution when predictors are correlated. The Shapley decomposition
averages each barrier's marginal contribution across every ordering - 2**7 = 128 subset
regressions - giving non-negative parts that sum to the R2 the block adds. That is what makes
"share of the explained gap" a meaningful quantity.

Two direction columns, because they can disagree. `marginal` is the sign of the raw correlation,
one barrier at a time. `in_model` is the sign of the coefficient, holding the other six barriers
and the year constant - the same model the Shapley share comes from, so it is the one that belongs
beside it.

Where they differ, the barrier correlates one way on its own and the other way once the rest is
controlled. Legal uncertainty is the clearest case: positively correlated with adoption by itself,
because richer countries have both more adoption and more legal debate, but negatively signed once
the other barriers are held fixed.

"WITH adoption" still means cited more where adoption is higher - a composition effect rather than
a driver, and not something to fix.

In [ ]:
result = ba.analyse(firm)          # ~8s: fits every tier and bootstraps each

rows = []
for tier, model in result["tiers"].items():
    for code, b in model["barriers"].items():
        rows.append({"tier": ba.TIER_LABELS[tier], "barrier": ba.SHORT_LABELS[code],
                     "share_pct": b["share"], "ci_low": b["interval"][0],
                     "ci_high": b["interval"][1],
                     "in_model": "against" if b["sign_in_model"] == -1 else "WITH adoption",
                     "coefficient": b["coefficient"],
                     "marginal": "against" if b["sign"] == -1 else "WITH adoption",
                     "cited_eu27_pct": b["exposure_eu27"],
                     "tier_stable": model["published"]})

shares = pd.DataFrame(rows).sort_values(["tier", "share_pct"], ascending=[True, False])
shares

## 6 - Does the ranking survive resampling?

The output is a ranking, so the ranking is what has to hold up. Whole **countries** are resampled,
not rows - four years of one country are not four independent observations.

A tier whose leading barrier leads in fewer than 60% of draws is flagged unstable. It is still
reported; it simply must not be presented as settled.

In [ ]:
for tier, model in result["tiers"].items():
    lead = max(model["barriers"].items(), key=lambda kv: kv[1]["share"])
    mark = "stable" if model["published"] else "NOT STABLE - top place is a tie"
    print(f"{ba.TIER_LABELS[tier]:18s} leads {model['lead_share']:5.0%} of draws "
          f"({ba.SHORT_LABELS[lead[0]]})   {mark}")

## 7 - Read the model country by country

Not a model per country - see step 2. This applies the tier model to each country:

```
expected     = intercept + year effect + (tier-average barriers) . beta
contribution = beta_j * (this country's barrier_j - the tier average)
actual       = expected + sum(contributions) + unexplained
```

So a country's gap decomposes into the part each barrier accounts for, plus a remainder. The
remainder is reported rather than hidden - it is usually large, and pretending otherwise would
overstate what seven survey questions can explain.

In [ ]:
country = ba.by_country(firm)

one = country[(country.geo == "IT") & (country.year == "2025")
              & (country.tier_code == "SMALL_10_49")]
print(f"Italy, small firms, 2025: actual {one.actual_adoption_pct.iloc[0]}%, "
      f"expected {one.expected_adoption_pct.iloc[0]}%, gap {one.gap_pp.iloc[0]:+.1f}pp")
print(f"  barriers account for {one.contribution_pp.sum():+.2f}pp, "
      f"unexplained {one.unexplained_pp.iloc[0]:+.2f}pp")

one[["barrier", "exposure_pct", "tier_mean_pct", "deviation_pp", "contribution_pp"]] \
    .sort_values("contribution_pp")

### Which countries the model reads well, and which it does not

A large unexplained remainder means the barriers say little about that country - worth knowing
before quoting a contribution for it.

In [ ]:
latest = country[country.year == country.year.max()]
fit = (latest.groupby(["tier", "geo"])
       .agg(gap_pp=("gap_pp", "first"), unexplained_pp=("unexplained_pp", "first"))
       .assign(abs_unexplained=lambda d: d.unexplained_pp.abs())
       .sort_values("abs_unexplained"))

print("best explained")
display(fit.head(5))
print("worst explained")
fit.tail(5)

### EU27 and all countries

Two aggregate rows are scored through the same model and marked `in_model = False`:

- **`EU27_2020`** - Eurostat's published aggregate, weighted by how many enterprises each country
  has.
- **`ALL_MEAN`** - a plain mean of the countries in the model, every country counting equally.

Neither is *fitted*. An aggregate is one row per year, not a sample, and EU27 in particular is a
weighted combination of the very rows it would otherwise sit beside - including it would let the
same countries vote twice.

Two things to look for below. The gap between EU27 and ALL_MEAN is the weighting: adoption tracks
economy size, so a plain country average sits above the enterprise-weighted one. And **ALL_MEAN's
unexplained remainder is exactly zero** - OLS residuals average to zero within each year, so the
composite of all countries is reproduced perfectly. That is a check on the arithmetic, not a
finding: if it were not zero, something upstream would be wrong.

In [ ]:
agg = country[(~country.in_model) & (country.tier_code == "SMALL_10_49")
              & (country.year == country.year.max())]

summary = (agg.groupby("geo")
           .agg(actual=("actual_adoption_pct", "first"),
                expected=("expected_adoption_pct", "first"),
                gap_pp=("gap_pp", "first"),
                contributions_pp=("contribution_pp", "sum"),
                unexplained_pp=("unexplained_pp", "first")))
print("Small firms, latest year")
display(summary.round(2))

print("\nEU27 gap, split by barrier")
agg[agg.geo == "EU27_2020"][["barrier", "exposure_pct", "tier_mean_pct",
                             "deviation_pp", "contribution_pp"]] \
    .sort_values("contribution_pp", ascending=False)

## 8 - Write the CSVs

The same two files `analyse_barriers.py` writes. **Close them in Excel first** - Windows locks an
open CSV and the write will fail.

In [ ]:
import json
import analyse_barriers as runner

out = ROOT / "data" / "analysis"
out.mkdir(parents=True, exist_ok=True)

names = json.loads((ROOT / "data" / "processed" / "firm_level.datamap.json")
                   .read_text(encoding="utf-8"))["columns"]["indicator"]["codes"]

summary = pd.DataFrame(runner.rows_from(result, names, draws=ba.BOOTSTRAP_DRAWS))
# Same helper the runner uses, so the two paths cannot drift apart.
country_out = runner.enrich_country(country, result)

# Windows locks a CSV that is open in Excel or an editor. Losing the run to a
# file handle after fitting every model would be a poor trade, so say which
# file and carry on rather than raising.
for frame, name in ((summary, "barrier_importance.csv"),
                    (country_out, "barrier_contributions_by_country.csv")):
    try:
        frame.to_csv(out / name, index=False, encoding="utf-8")
        print(f"wrote {name} ({len(frame)} rows, {len(frame.columns)} columns)")
    except PermissionError:
        print(f"SKIPPED {name} - it is open in another program. Close it and re-run this cell.")

print("destination:", out)

## How to read the result

- **`share_pct` divides the R2 the barriers add, not all of adoption.** For small firms that is
  0.27 - so a barrier holding 65% of the share explains 65% of *that*, not of everything.
- **`cited_eu27_pct` against `share_pct` is the interesting comparison.** For small firms, cost
  takes 64.7% of the explained gap while being cited by 38.8%; lack of expertise takes 7.2% while
  being cited by 70.9%. **What firms cite most is not what separates them.**
- **`direction = WITH adoption` marks a composition effect,** not something to fix.
- **Large firms are not settled** - cost, expertise and data quality are effectively tied, which
  is why that tier is flagged rather than ranked.
- Nothing here is causal.